# Resume / Candidate Screening System

This project builds an intelligent system to automatically screen, score, and rank resumes based on a given job description using NLP and Machine Learning techniques.

## Problem Statement

Recruiters often receive hundreds of resumes for a single job role. Manually screening them is:

- Time-consuming  
- Inconsistent  
- Error-prone  

This project aims to automate resume screening by building a system that:

- Extracts skills from resumes  
- Matches them with job requirements  
- Ranks candidates based on relevance  
- Identifies missing skills  

## Objectives

- Perform resume text preprocessing  
- Extract important skills using NLP  
- Compare resumes with a job description  
- Rank candidates based on job fit  
- Identify skill gaps  

## Tools & Technologies Used

- Python  
- Pandas, NumPy  
- NLTK (text preprocessing)  
- Scikit-learn (TF-IDF, similarity)  
- Jupyter Notebook  

In [1]:
!pip install spacy

In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------- ----------------------------- 3.4/12.8 MB 22.4 MB/s eta 0:00:01
     ------------------ --------------------- 5.8/12.8 MB 14.9 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 14.6 MB/s eta 0:00:01
     ---------------------------------- ---- 11.3/12.8 MB 14.1 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 13.6 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
# Basic Libraries
import pandas as pd
import numpy as np
import re
import os

# NLP Libraries
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load English model
nlp = spacy.load("en_core_web_sm")

# NLTK stopwords (download if not present)
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to C:\Users\Chinmayi
[nltk_data]     MH\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Dataset Description

The dataset contains resumes with the following columns:

- **ID** – Unique identifier  
- **Resume_str** – Resume text (used for analysis)  
- **Resume_html** – HTML version (ignored)  
- **Category** – Job category  

We use the **Resume_str** column for processing and analysis.

In [4]:
dataset_path = r"C:\Users\Chinmayi MH\Downloads\Resume\resume.csv"

df = pd.read_csv(dataset_path)
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


## Methodology

The system follows these steps:

1. **Text Cleaning**
   - Lowercasing
   - Removing special characters
   - Removing stopwords  

2. **Skill Extraction**
   - Identify predefined skills from resume text  

3. **Job Description Processing**
   - Extract required skills from job description  

4. **Similarity Calculation**
   - Use TF-IDF vectorization  
   - Compute cosine similarity  

5. **Skill Matching**
   - Compare resume skills with job requirements  

6. **Final Scoring**
   - Combine similarity score and weighted skill score  

7. **Ranking**
   - Rank candidates based on final score  

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply cleaning on Resume_str
df['cleaned_text'] = df['Resume_str'].apply(clean_text)

df[['Resume_str', 'cleaned_text']].head()

,Resume_str,cleaned_text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administratormarketing associate hr adminis...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versati...
2,HR DIRECTOR Summary Over 2...,hr director summary 20 years experience recrui...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights hr skills hr depar...


In [6]:
skills_list = ['python', 'java', 'c++', 'machine learning', 'nlp', 
               'sql', 'excel', 'deep learning', 'tensorflow', 'pytorch']

def extract_skills(text):
    tokens = text.split()  # simpler & faster than spaCy
    found_skills = [skill for skill in skills_list if skill in text]
    return found_skills

df['skills'] = df['cleaned_text'].apply(extract_skills)

df[['cleaned_text', 'skills']].head()

,cleaned_text,skills
0,hr administratormarketing associate hr adminis...,[]
1,hr specialist us hr operations summary versati...,[]
2,hr director summary 20 years experience recrui...,[excel]
3,hr specialist summary dedicated driven dynamic...,[excel]
4,hr manager skill highlights hr skills hr depar...,[excel]


In [7]:
job_description = """
Looking for a Data Scientist with skills in Python, Machine Learning, NLP, SQL.
Experience in Deep Learning, TensorFlow or PyTorch is preferred.
"""

clean_jd = clean_text(job_description)
jd_skills = extract_skills(clean_jd)

print("Job Description Skills:", jd_skills)

Job Description Skills: ['python', 'machine learning', 'nlp', 'sql', 'deep learning', 'tensorflow', 'pytorch']


In [8]:
vectorizer = TfidfVectorizer()

all_texts = df['cleaned_text'].tolist() + [clean_jd]

tfidf_matrix = vectorizer.fit_transform(all_texts)

similarity_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]

df['similarity_score'] = similarity_scores

df.head()

,ID,Resume_str,Resume_html,Category,cleaned_text,skills,similarity_score
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administratormarketing associate hr adminis...,[],0.006255
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist us hr operations summary versati...,[],0.001052
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary 20 years experience recrui...,[excel],0.001587
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist summary dedicated driven dynamic...,[excel],0.004101
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlights hr skills hr depar...,[excel],0.002080


In [9]:
def missing_skills(resume_skills, jd_skills):
    return list(set(jd_skills) - set(resume_skills))

df['missing_skills'] = df['skills'].apply(lambda x: missing_skills(x, jd_skills))

# Rank candidates
df = df.sort_values(by='similarity_score', ascending=False).reset_index(drop=True)

df[['ID', 'Category', 'skills', 'missing_skills', 'similarity_score']].head(10)

,ID,Category,skills,missing_skills,similarity_score
0,12011623,ENGINEERING,"[python, machine learning, sql, excel]","[pytorch, tensorflow, nlp, deep learning]",0.124041
1,12777487,ARTS,[],"[machine learning, tensorflow, deep learning, ...",0.118079
2,34953092,BANKING,"[python, machine learning, sql]","[pytorch, tensorflow, nlp, deep learning]",0.117103
3,62994611,AGRICULTURE,"[python, java, sql, excel, tensorflow]","[pytorch, machine learning, nlp, deep learning]",0.113153
4,21156767,CONSULTANT,"[python, java, machine learning, sql]","[pytorch, tensorflow, nlp, deep learning]",0.106390
5,18835363,DESIGNER,[],"[machine learning, tensorflow, deep learning, ...",0.100013
6,30863060,CONSULTANT,"[java, sql, excel]","[machine learning, deep learning, pytorch, nlp...",0.095286
7,36206485,TEACHER,[excel],"[machine learning, deep learning, pytorch, sql...",0.086183
8,22946204,AUTOMOBILE,"[python, java, machine learning, sql, excel]","[pytorch, tensorflow, nlp, deep learning]",0.081166
9,18448085,AUTOMOBILE,"[python, sql, excel]","[machine learning, deep learning, pytorch, nlp...",0.078935


## Skill Weighting Strategy

Not all skills are equally important.  

We assign weights based on importance:

- High Importance: Python, Machine Learning, NLP, Deep Learning  
- Medium Importance: TensorFlow, PyTorch, SQL  
- Low Importance: Java, Excel  

This ensures that critical skills influence the ranking more significantly.

In [10]:
skill_weights = {
    'python': 3,
    'machine learning': 3,
    'nlp': 3,
    'deep learning': 3,
    'tensorflow': 2,
    'pytorch': 2,
    'sql': 2,
    'java': 1,
    'excel': 1
}

## Scoring Logic

The final score is calculated as:

Final Score = (0.6 × Similarity Score) + (0.4 × Skill Score)

- **Similarity Score** → Measures textual relevance using TF-IDF  
- **Skill Score** → Measures how many required skills match  

This hybrid approach improves ranking accuracy.

In [11]:
def calculate_skill_score(resume_skills, jd_skills):
    score = 0
    total = 0
    
    for skill in jd_skills:
        weight = skill_weights.get(skill, 1)
        total += weight
        if skill in resume_skills:
            score += weight
            
    return score / total if total != 0 else 0

## Results

The system outputs:

- Ranked list of candidates  
- Extracted skills for each candidate  
- Missing skills compared to job description  
- Final score indicating job fit  

Top candidates have:
- Higher similarity scores  
- More matching skills  
- Fewer missing skills  

In [12]:
df['skill_score'] = df['skills'].apply(lambda x: calculate_skill_score(x, jd_skills))

# Final combined score
df['final_score'] = (0.6 * df['similarity_score']) + (0.4 * df['skill_score'])

# Re-rank
df = df.sort_values(by='final_score', ascending=False).reset_index(drop=True)

df[['ID', 'skills', 'missing_skills', 'similarity_score', 'skill_score', 'final_score']].head(10)

,ID,skills,missing_skills,similarity_score,skill_score,final_score
0,12011623,"[python, machine learning, sql, excel]","[pytorch, tensorflow, nlp, deep learning]",0.124041,0.444444,0.252202
1,34953092,"[python, machine learning, sql]","[pytorch, tensorflow, nlp, deep learning]",0.117103,0.444444,0.248039
2,21156767,"[python, java, machine learning, sql]","[pytorch, tensorflow, nlp, deep learning]",0.106390,0.444444,0.241612
3,50328713,"[python, machine learning, sql, tensorflow]","[pytorch, nlp, deep learning]",0.029733,0.555556,0.240062
4,22946204,"[python, java, machine learning, sql, excel]","[pytorch, tensorflow, nlp, deep learning]",0.081166,0.444444,0.226477
5,62994611,"[python, java, sql, excel, tensorflow]","[pytorch, machine learning, nlp, deep learning]",0.113153,0.388889,0.223447
6,21297521,"[java, nlp, sql, excel, deep learning]","[pytorch, machine learning, python, tensorflow]",0.035404,0.444444,0.199020
7,18448085,"[python, sql, excel]","[machine learning, deep learning, pytorch, nlp...",0.078935,0.277778,0.158472
8,42156237,"[python, java, sql, excel]","[machine learning, deep learning, pytorch, nlp...",0.074471,0.277778,0.155794
9,18067556,"[python, sql, excel]","[machine learning, deep learning, pytorch, nlp...",0.066297,0.277778,0.150889


In [13]:
for i in range(5):
    print(f"\nCandidate Rank #{i+1}")
    print("ID:", df.loc[i, 'ID'])
    print("Skills:", df.loc[i, 'skills'])
    print("Missing Skills:", df.loc[i, 'missing_skills'])
    print("Final Score:", round(df.loc[i, 'final_score'], 3))


Candidate Rank #1
ID: 12011623
Skills: ['python', 'machine learning', 'sql', 'excel']
Missing Skills: ['pytorch', 'tensorflow', 'nlp', 'deep learning']
Final Score: 0.252

Candidate Rank #2
ID: 34953092
Skills: ['python', 'machine learning', 'sql']
Missing Skills: ['pytorch', 'tensorflow', 'nlp', 'deep learning']
Final Score: 0.248

Candidate Rank #3
ID: 21156767
Skills: ['python', 'java', 'machine learning', 'sql']
Missing Skills: ['pytorch', 'tensorflow', 'nlp', 'deep learning']
Final Score: 0.242

Candidate Rank #4
ID: 50328713
Skills: ['python', 'machine learning', 'sql', 'tensorflow']
Missing Skills: ['pytorch', 'nlp', 'deep learning']
Final Score: 0.24

Candidate Rank #5
ID: 22946204
Skills: ['python', 'java', 'machine learning', 'sql', 'excel']
Missing Skills: ['pytorch', 'tensorflow', 'nlp', 'deep learning']
Final Score: 0.226


In [14]:
def explain_candidate(row):
    return f"""
    Candidate {row['ID']} ranked with score {round(row['final_score'],3)}
    - Matching Skills: {row['skills']}
    - Missing Skills: {row['missing_skills']}
    """

print(explain_candidate(df.iloc[0]))


    Candidate 12011623 ranked with score 0.252
    - Matching Skills: ['python', 'machine learning', 'sql', 'excel']
    - Missing Skills: ['pytorch', 'tensorflow', 'nlp', 'deep learning']
    


## Conclusion

This project successfully demonstrates how Machine Learning and NLP can be used to automate resume screening.

The system:
- Reduces manual effort  
- Improves consistency in hiring  
- Provides explainable candidate rankings  

This approach is similar to real-world HR-tech systems used in recruitment platforms.

## Future Improvements

- Use advanced NLP models (BERT, transformers)  
- Improve skill extraction using Named Entity Recognition  
- Add a user interface (web app)  
- Include real-time resume upload and analysis  
- Enhance scoring with experience and education  